#  ______________________________________________________________________________________________________

# DESIGN DOCUMENT

# 📋 Executive Summary
This document describes the architecture of a goal-oriented multi-agent system built with CrewAI to handle customer queries for a retail bank operating under RBI jurisdiction. The system routes incoming customer messages through four specialized agents — each with a clear role, tightly scoped goal, and explicit input/output contract — to either auto-resolve routine queries or escalate high-risk cases to the correct human team.

In [1]:
from IPython.display import HTML

# Example: Wrapping text diagram in a stylized HTML container
diagram_text = """
🧠 Pipeline for Customer Support Automation: 
Customer Message
      │
      ▼
[Agent 1: Intent Classification]
      │
      ├─ intent + urgency_signal
      ▼
[Agent 2: Policy Reasoning]
      │
      ├─ auto_resolvable=true ──────────► [Agent 3: Response Drafting]
      │                                          │
      └─ auto_resolvable=false                   │
             │                                   │
             ▼                                   ▼
      [Agent 4: Risk & Escalation] ◄─────────────┘
             │
             ├─ escalate=false → deliver Agent 3 draft as final response
             └─ escalate=true  → route to appropriate team + notify customer
"""

html_content = f"""
<div style="border: 1px solid #ccc; padding: 10px; background-color: #f9f9f9; font-family: monospace;">
    <pre>{diagram_text}</pre>
</div>
"""
display(HTML(html_content))

# 🛠️ Key Design Principles

*   **Policy-first:** Every resolution decision is grounded in a named **RBI circular**, rather than heuristics or probabilistic logic.
*   **Agent 4 (Escalation):** Always runs. Even auto-resolved cases are reviewed by this agent before final delivery to ensure a "human-in-the-loop" equivalent check.
*   **Escalation Tiers:** Routing is granular rather than a binary yes/no. Cases are directed to specific functional units:
    *   `fraud_team`
    *   `l2_operations`
    *   `l1_support`
*   **No RAG:** Agent 2 utilizes a **structured policy engine** instead of Retrieval-Augmented Generation, ensuring behavior is 100% predictable, deterministic, and auditable.


# 🏦 Banking AI Agent Architecture

### 🤖 Agent 1: Intent Classification Agent

| Field | Details |
|-------|---------|
| **Role** | Banking Intent Classification Specialist |
| **Goal** | Classify the customer's message into one of 8 categories and extract entities (amount, urgency) |
| **Input** | Raw customer message text (string) |
| **Output** | JSON: `{ intent, urgency_signal, entities: { amount, timestamp }, raw_message }` |
| **Logic** | Keyword-pattern matching across 8 intents; regex-based amount extraction (Rs/INR/lakh); urgency keyword scan |

---

### 🤖 Agent 2: Banking Rules & Policy Reasoning Agent

| Field | Details |
|-------|---------|
| **Role** | RBI Policy & Compliance Reasoning Specialist |
| **Goal** | Apply the correct RBI guideline to the classified intent; determine auto-resolvability and risk flags |
| **Input** | JSON from Agent 1 (intent classification output) |
| **Output** | JSON: `{ policy_name, policy_summary, auto_resolvable, resolution_path, risk_flags, sla_hours }` |
| **Logic** | Stateless policy dictionary keyed by intent. High-value override: amount ≥ Rs 1 lakh on resolvable intents triggers `auto_resolvable=False` |

---

### 🤖 Agent 3: Response Drafting Agent

| Field | Details |
|-------|---------|
| **Role** | Customer Response Drafting Specialist |
| **Goal** | Draft a clear, empathetic, policy-compliant response with actionable next steps and SLA commitment |
| **Input** | JSON from Agent 2 (policy output) — only invoked when `auto_resolvable=True` |
| **Output** | JSON: `{ greeting, body, next_steps[], policy_reference, sla_commitment, draft_complete }` |
| **Logic** | Intent-specific templates with dynamic amount, SLA, and resolution path interpolation. Returns `draft_complete=False` if skipped |

---

### 🤖 Agent 4: Risk & Escalation Agent

| Field | Details |
|-------|---------|
| **Role** | Risk Assessment & Escalation Decision Specialist |
| **Goal** | Final gatekeeper: apply hard and soft escalation rules; route to correct tier; compose final customer response |
| **Input** | Combined JSON from all prior agents (intent + policy + draft) |
| **Output** | JSON: `{ escalate, escalation_tier, escalation_reason[], internal_actions[], customer_facing_response, resolution_status }` |
| **Logic** | Hard rules: fraud/security → fraud_team; wrong_transfer → l2_operations (legal if >1L). Soft rules: high-value+non-resolvable → l2; non-resolvable+urgent → l1; non-resolvable → l1_low_priority |


# 🤝 Task Flow & Agent Handoffs
Tasks are defined with explicit context dependencies using CrewAI's context=[] parameter. Each task receives the complete JSON output of its predecessor(s) — not parsed fields, the full JSON string. This ensures no information is lost in handoffs.

| Task	| Handoff Data Passed Forward |
|-------|---------------------------- |
| Task 1 → Task 2	| intent, urgency_signal, entities.amount, raw_message |
| Task 2 → Task 3	| policy_name, auto_resolvable, risk_flags, sla_hours, resolution_path, amount |
| Tasks 1+2+3 → Task 4	| Full combined JSON: intent + policy + draft_response object |

Agent 4 explicitly:
- Receives context from **all three prior tasks**.
- Constructs a unified context object before calling the escalation engine. 

**This design means Agent 4 can override even a well-drafted Agent 3 response if risk flags demand escalation.**

# 🧗 Escalation Logic
Hard rules trigger unconditionally based on intent. Soft rules apply when hard rules do not match, combining urgency and auto-resolvability signals. Escalation tiers map directly to bank operations teams.

| Trigger | Condition |	Rule Type | Escalation Tier	Rationale |
| ------- | --------- | --------- | ------------------------- |
| Intent = fraud	| Hard	| fraud_team	| Zero-tolerance — RBI 2017-18/15 mandatory | 
| Intent = security	| Hard	| fraud_team	| Account takeover — CERT-In notification in 6h |
| Intent = wrong_transfer	| Hard	| l2_operations	| 48-hour NPCI reversal SLA time-critical |
| wrong_transfer + amount >= 1L	| Hard | l2_operations_legal | RBI Ombudsman notification may apply |
| Amount >= 1L + not auto-resolvable | Soft | l2_operations | High-value dispute exceeds auto threshold |
| Not auto-resolvable + urgency | Soft | l1_support | Policy gap + urgency — human judgment needed |
| Not auto-resolvable (no urgency) | Soft | l1_support (low) | Needs human policy review, non-urgent |
| All other cases | Pass | None (auto-resolved) | Agent 3 draft delivered as final response | 


# ⚖️ RBI Policy Reference
Agent 2's policy engine maps each intent to a named RBI circular or master direction. This is not RAG — policies are hardcoded as a deterministic dictionary, ensuring consistent and auditable behaviour in a regulated environment.

| Intent Category | RBI Policy / Guideline |
| --------------- | ---------------------- |
| fraud | RBI Master Direction – Limiting Liability of Customers (RBI/2017-18/15) |
| security | RBI Cybersecurity Framework for Banks (DBS.CO/CSITE/BC.11) + IT Act 2000 |
| wrong_transfer | Payment & Settlement Systems Act 2007 + NPCI UPI Dispute Guidelines |
| transaction | RBI Circular on Failed Transactions (DPSS.CO.PD.No.629/02.01.014/2019-20) |
| loan | RBI Fair Practices Code for Lenders + Credit Information Companies Act 2005 |
| card | RBI Master Direction on Card Transactions (RBI/2021-22/74) |
| account | RBI KYC Master Direction 2016 + Banking Ombudsman Scheme 2006 |
| general | Banking Ombudsman Scheme 2006 |


#  ______________________________________________________________________________________________________

# CODE

In [2]:
import sys
print(sys.executable)  # should show .../miniforge/base/envs/crunch/bin/python

from crewai import Agent, Task, Crew, Process, LLM
from dotenv import load_dotenv
print("all imports ok")

/opt/homebrew/Caskroom/miniforge/base/envs/crunch/bin/python
all imports ok


In [7]:
import os

from banking_crew import run_crew, run_comparison, SAMPLE_QUERIES

# Single query, Gemini
result = run_crew(SAMPLE_QUERIES[0], "gemini")
print(result)

# Single query, Claude
result = run_crew(SAMPLE_QUERIES[5], "claude")
print(result)

# Side-by-side comparison
results = run_comparison(SAMPLE_QUERIES[5])
print("GEMINI:", results["gemini"])
print("CLAUDE:", results["claude"])

ModuleNotFoundError: No module named 'banking_crew'

In [9]:
import os
os.chdir(os.path.dirname(os.path.abspath("setup_notebook.ipynb")))

from banking import run_crew, run_comparison, SAMPLE_QUERIES

In [10]:
print(f"Loaded {len(SAMPLE_QUERIES)} sample queries")

Loaded 10 sample queries


In [11]:
SAMPLE_QUERIES

['What is my current account balance?',
 'I want to apply for a personal loan of Rs 5 lakhs.',
 'My debit card was declined at the POS terminal today.',
 'Why was my loan application rejected last week?',
 'My UPI payment of Rs 500 failed but money was debited from my account.',
 'Someone made 3 unauthorized transactions from my credit card totalling Rs 45000 which I did not authorize.',
 'I accidentally transferred Rs 2 lakhs to the wrong account. Please help urgently!',
 'There are 5 suspicious logins to my net banking from unknown devices in the last hour.',
 "My salary of Rs 85000 hasn't been credited this month and it is already the 5th.",
 'I received an OTP I did not request — I think someone is trying to hack my account.']

In [ ]:
result = run_crew(SAMPLE_QUERIES[5], "claude")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  3cd3a25c-045a-4c68-9f1f-35dffce4f027                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Classify this message:                                                                                   │
│                                                                                                                 │
│  "Someone made 3 unauthorized transactions from my credit card totalling Rs 45000 which I did not authorize."   │
│                                                                                                                 │
│  Use intent_classifier. Return full JSON.                                                                       │
│  ID: 5bcfaded-85e2-42b3-aa93-0f2c64df9969                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Banking Intent Classification Specialist                                                                │
│                                                                                                                 │
│  Task: Classify this message:                                                                                   │
│                                                                                                                 │
│  "Someone made 3 unauthorized transactions from my credit card totalling Rs 45000 which I did not authorize."   │
│                                                                                                                 │
│  Use intent_classifier. Return full JSON.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Anthropic API call failed: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-3-5-haiku-20241022'}, 'request_id': 'req_011Cam8WVsdxJv3tcAUmM7qj'}


An unknown error occurred. Please check the details below.
Error details: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-3-5-haiku-20241022'}, 'request_id': 'req_011Cam8WVsdxJv3tcAUmM7qj'}
An unknown error occurred. Please check the details below.
Error details: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-3-5-haiku-20241022'}, 'request_id': 'req_011Cam8WVsdxJv3tcAUmM7qj'}


...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

NotFoundError: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-3-5-haiku-20241022'}, 'request_id': 'req_011Cam8WZJGHXVuq9PQbn1Uw'}

ERROR:crewai.events.listeners.tracing.trace_batch_manager:Failed to send events: 404. Response: {"error":"Couldn't find EphemeralTraceBatch with [WHERE \"ephemeral_trace_batches\".\"ephemeral_trace_id\" = $1]","message":"Trace batch not found"}. Events will be lost.




╭───────────────────────── 🔍 Local Traces Collected ──────────────────────────╮
│                                                                              │
│  📊 Your execution traces were collected locally!                            │
│                                                                              │
│  Unfortunately, we couldn't upload them to the server right now, but here's  │
│  what we captured:                                                           │
│  • 14 trace events                                                           │
│  • Execution duration: 16293ms                                               │
│  • Batch ID: ab679365-af63-4660-a442-1c7a5877dbdc                            │
│                                                                              │
│  ✅ Tracing has been enabled for future runs!                                │
│  Your preference has been saved. Future Crew/Flow executions will            │
│  automatically collect trac

In [ ]:
result = run_crew(SAMPLE_QUERIES[5], "gemini")
print(result)